# 06 Boolean Targeting Walkthrough

This notebook shows how the demo can be extended to support richer campaign targeting without turning Redis into a full query engine.

The practical split is:
- use Redis Sets for a high-recall candidate domain,
- use app-side logic for exact evaluation of more complicated boolean rules,
- rerank only the surviving campaigns.

That means we can support structured filters such as `all_of`, `any_of`, and `none_of` directly in the candidate-generation layer, while still leaving room for exact expression checks such as XOR or nested branch-specific negation.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise RuntimeError('Could not locate repo root')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.models import UserProfile
from data.common import read_jsonl

DATASET_DIR = REPO_ROOT / 'data' / 'generated' / 'synthetic'
users = [UserProfile.model_validate(row) for row in read_jsonl(DATASET_DIR / 'users.jsonl')]
sample_user = max(users, key=lambda user: len(user.segments))
sample_user

## Example Campaign Targeting Shapes

These example campaigns use the same segment vocabulary as the synthetic demo, but they carry richer targeting metadata than the current API implementation.

Two patterns are shown:
- a structured form that Redis can mostly evaluate with set algebra,
- a more complex `exact_expression` that should be evaluated in the app after retrieval.

In [ ]:
campaigns = [
    {
        'campaign_id': 'bool_001',
        'geo': ['US'],
        'device': ['iOS', 'Android'],
        'all_of': ['camping_high'],
        'any_of': ['travel_high', 'family_high'],
        'none_of': ['gaming_high'],
        'exact_expression': None,
        'notes': 'Fully expressible with Redis set algebra',
    },
    {
        'campaign_id': 'bool_002',
        'geo': ['US'],
        'device': ['iOS'],
        'all_of': ['home_improvement_high'],
        'any_of': ['pet_care_high', 'family_high'],
        'none_of': ['luxury_high'],
        'exact_expression': None,
        'notes': 'Another pure all_of / any_of / none_of rule',
    },
    {
        'campaign_id': 'bool_003',
        'geo': ['US'],
        'device': ['iOS'],
        'all_of': [],
        'any_of': ['camping_high', 'gaming_high', 'travel_high'],
        'none_of': [],
        'exact_expression': {
            'or': [
                {'xor': ['camping_high', 'gaming_high']},
                {'and': ['travel_high', {'not': 'luxury_high'}]},
            ]
        },
        'notes': 'Candidate domain comes from positive atoms; exact logic runs in app',
    },
]

pd.DataFrame(campaigns)[['campaign_id', 'all_of', 'any_of', 'none_of', 'notes']]

## Build Inverted Indexes For The Structured Filters

For the `all_of` / `any_of` / `none_of` fields, the indexing strategy is the same as the current demo:
- each targeting bucket gets an `idx:segment:<bucket>` set,
- each campaign ID is inserted into the sets for the buckets it references.

The difference is that `any_of` and `none_of` are now first-class parts of the targeting schema.

In [ ]:
indexes = defaultdict(set)
for campaign in campaigns:
    for geo in campaign['geo']:
        indexes[f'idx:geo:{geo}'].add(campaign['campaign_id'])
    for device in campaign['device']:
        indexes[f'idx:device:{device}'].add(campaign['campaign_id'])
    for segment in set(campaign['all_of']) | set(campaign['any_of']) | set(campaign['none_of']):
        indexes[f'idx:segment:{segment}'].add(campaign['campaign_id'])

pd.DataFrame(
    [
        {'key': key, 'members': sorted(values)}
        for key, values in sorted(indexes.items())
    ]
)

## Coarse Candidate Domain With Set Algebra

A practical Redis plan for structured targeting is:
- intersect `geo` and `device`,
- intersect all `all_of` segment sets,
- union the `any_of` segment sets and intersect that union into the positive pool,
- union the `none_of` segment sets and subtract them from the positive pool.

The helper below performs that logic in memory and also emits the equivalent Redis command plan.

In [ ]:
def structured_candidate_domain(user: UserProfile, campaign: dict, indexes: dict[str, set[str]]) -> tuple[set[str], list[str]]:
    commands: list[str] = []
    universe = set(campaign['campaign_id'] for campaign in campaigns)

    positive = universe.copy()
    base_keys = [f"idx:geo:{user.geo}", f"idx:device:{user.device}"]
    commands.append('SINTER ' + ' '.join(base_keys))
    for key in base_keys:
        positive &= indexes.get(key, set())

    for segment in campaign['all_of']:
        key = f'idx:segment:{segment}'
        commands.append(f'SINTER current {key}')
        positive &= indexes.get(key, set())

    if campaign['any_of']:
        keys = [f'idx:segment:{segment}' for segment in campaign['any_of']]
        commands.append('SUNION ' + ' '.join(keys))
        any_pool = set().union(*(indexes.get(key, set()) for key in keys))
        commands.append('SINTER current __any_pool__')
        positive &= any_pool

    if campaign['none_of']:
        keys = [f'idx:segment:{segment}' for segment in campaign['none_of']]
        commands.append('SUNION ' + ' '.join(keys))
        none_pool = set().union(*(indexes.get(key, set()) for key in keys))
        commands.append('SDIFF current __none_pool__')
        positive -= none_pool

    return positive, commands


structured_rows = []
for campaign in campaigns:
    domain, command_plan = structured_candidate_domain(sample_user, campaign, indexes)
    structured_rows.append(
        {
            'campaign_id': campaign['campaign_id'],
            'candidate_domain': sorted(domain),
            'command_plan': command_plan,
        }
    )

pd.DataFrame(structured_rows)

## Exact App-Side Evaluation For More Complicated Rules

Structured fields are enough for many targeting use cases, but they are not enough for everything.

Example: `(camping_high XOR gaming_high) OR (travel_high AND NOT luxury_high)`

A good production pattern is:
- use Redis to over-approximate the candidate domain with high recall,
- run the exact expression in the app on the small candidate set,
- only rerank campaigns that pass the exact test.

The exact evaluator below operates over the user's segment set.

In [ ]:
def evaluate_expression(node: object, user_segments: set[str]) -> bool:
    if isinstance(node, str):
        return node in user_segments
    if not isinstance(node, dict):
        raise TypeError(f'Unsupported node: {node!r}')
    if 'and' in node:
        return all(evaluate_expression(child, user_segments) for child in node['and'])
    if 'or' in node:
        return any(evaluate_expression(child, user_segments) for child in node['or'])
    if 'not' in node:
        return not evaluate_expression(node['not'], user_segments)
    if 'xor' in node:
        children = [evaluate_expression(child, user_segments) for child in node['xor']]
        return sum(children) == 1
    raise ValueError(f'Unsupported expression node: {node}')


user_segments = set(sample_user.segments)
pd.Series(
    {
        'user_id': sample_user.user_id,
        'segments': sorted(user_segments),
        'bool_003_exact_match': evaluate_expression(campaigns[2]['exact_expression'], user_segments),
    }
)

## Putting The Two Stages Together

For the more complicated campaign, the coarse domain is intentionally broader than the final exact rule.
That is the whole point: Redis narrows the search space quickly, then the app restores exactness.

In [ ]:
combined_rows = []
for campaign in campaigns:
    domain, _ = structured_candidate_domain(sample_user, campaign, indexes)
    exact_match = True if campaign['exact_expression'] is None else evaluate_expression(campaign['exact_expression'], user_segments)
    combined_rows.append(
        {
            'campaign_id': campaign['campaign_id'],
            'in_candidate_domain': campaign['campaign_id'] in domain,
            'passes_exact_expression': exact_match,
            'eligible_for_rerank': (campaign['campaign_id'] in domain) and exact_match,
        }
    )

pd.DataFrame(combined_rows)

## Practical Takeaway

This is the extension path that fits the existing demo architecture:
- add `all_of`, `any_of`, and `none_of` to campaign metadata,
- generate Redis candidate domains with `SINTER`, `SUNION`, and `SDIFF`,
- optionally keep a permissive domain for campaigns with branch-local logic,
- evaluate any nested boolean expression in the app,
- rerank only the surviving campaigns.

That is enough to tell a credible internal story about how the current set-based design could grow into richer ad-targeting logic.